# AM Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enph-479-edge-ai-stt/enph-479-edge-ai-stt/blob/feat/streamline-am-training/training/notebooks/am_training.ipynb)

Trains the acoustic model (3x256 unidirectional LSTM + CTC) on LibriSpeech train-clean-100 and prints greedy dev-clean CER after every epoch. **Runtime: T4 GPU.** Runs start to finish in one session, then saves the weights and the training log to a run folder on Google Drive.

Features are computed on the fly in the DataLoader workers. If GPU utilization in Colab's Resources panel stays low while the CPU is pegged, feature extraction is the bottleneck.

Specs (30-symbol vocab, feature constants) are provisional until `shared/specs` is frozen.

## 1. Pull the repo

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/enph-479-edge-ai-stt/enph-479-edge-ai-stt.git"
REPO_DIR = "/content/enph-479-edge-ai-stt"
BRANCH = "feat/streamline-am-training"  # TODO: back to "main" (and the badge) before the PR

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
# Always land on the tip of BRANCH, even if an earlier run left the clone on another branch.
git = ["git", "-C", REPO_DIR]
subprocess.run([*git, "fetch", "origin", BRANCH], check=True)
subprocess.run([*git, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
print(subprocess.run([*git, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Installs this checkout's package and its dependencies (jiwer, soundfile);
# torch and torchaudio come with Colab.
training_dir = os.path.join(REPO_DIR, "training")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", training_dir], check=True)

# The editable install only takes effect at interpreter start, so put src/ on the
# path for this already-running kernel.
sys.path.insert(0, os.path.join(training_dir, "src"))
importlib.invalidate_caches()

## 2. Mount Drive and configure the run

Mounting Drive here means the permission prompt comes up now, not after training. The results go to `MyDrive/enph479-stt/runs/<start time>/` at the end.

One epoch over train-clean-100 takes about 9 minutes on a T4, so 20 epochs is about 3 hours.

In [ ]:
import time

from google.colab import drive

from training.am.train import TrainConfig

drive.mount("/content/drive")
RUN_DIR = f"/content/drive/MyDrive/enph479-stt/runs/{time.strftime('%Y%m%d-%H%M')}"

cfg = TrainConfig(epochs=20)
print(RUN_DIR)
cfg

## 3. Data

Downloads train-clean-100 (6.3 GB) and dev-clean (337 MB) to `/content/data`, then computes CMVN from the training utterances.

In [ ]:
from training.am.dataset import LibriSpeechFeatures, compute_cmvn_over, list_utterances
from training.data.librispeech import download_subset

train_items = list_utterances(download_subset("train-clean-100", "/content/data"))
dev_items = list_utterances(download_subset("dev-clean", "/content/data"))
print(f"train {len(train_items)} utts, dev {len(dev_items)} utts")

# Every 10th training utterance is plenty for a global mean/std.
mean, std = compute_cmvn_over(train_items[::10], num_workers=cfg.num_workers)

# Each role takes a list of datasets in any format; train() pools each list.
datasets = {
    "train": [LibriSpeechFeatures(train_items, mean, std)],
    "test": [LibriSpeechFeatures(dev_items, mean, std)],
}

## 4. Train

Expect the blank wall first: test CER sits near 1.0 while the model outputs only blanks, then characters appear and it falls. Progress lines show s/batch; each epoch line shows train and eval time.

In [ ]:
from training.am.train import train

model = train(cfg, datasets, "train.log")

## 5. Save to Drive

Writes `am.pt` (the weights plus the CMVN mean/std the model's inputs were normalized with; the weights are useless without them) and `train.log` to the run folder, then flushes Drive so the upload finishes before the session ends.

In [ ]:
import shutil

import torch

os.makedirs(RUN_DIR, exist_ok=True)
torch.save({"model": model.state_dict(), "cmvn_mean": mean, "cmvn_std": std}, f"{RUN_DIR}/am.pt")
shutil.copy("train.log", RUN_DIR)
print(f"saved am.pt and train.log to {RUN_DIR}")
drive.flush_and_unmount()